# BIST Prodüksiyon Sinyal Sistemi
## Gerçek Veri · WFO Doğrulama · Kelly Pozisyon · ATR Stop-Loss

### Sistem Mimarisi

```
┌──────────────────────────────────────────────────────────────────┐
│  KATMAN 1 — VERİ                                                 │
│  yfinance → BIST OHLCV + XU100 endeks (gerçek tarihsel veri)    │
├──────────────────────────────────────────────────────────────────┤
│  KATMAN 2 — SİNYAL (WFO doğrulanmış)                            │
│  HMM Rejim + Adaptive Momentum → Ensemble sinyal (1=AL, 0=FLAT) │
├──────────────────────────────────────────────────────────────────┤
│  KATMAN 3 — RİSK YÖNETİMİ                                       │
│  Kelly Kriteri → Optimal pozisyon büyüklüğü (%)                 │
│  ATR Stop-Loss → Dinamik zarar durdurma fiyatı                  │
│  Circuit Breaker → Max günlük kayıp / Max DD limiti              │
├──────────────────────────────────────────────────────────────────┤
│  KATMAN 4 — ÇIKTI                                                │
│  Bugünkü sinyal | Lot sayısı | Stop fiyatı | R:R oranı           │
└──────────────────────────────────────────────────────────────────┘
```

**Neden bu kombinasyon?**
- WFO testi: HMM ve AdaptMom geçmiş dönemde en tutarlı performansı gösterdi
- Kelly: aynı strateji, yanlış lot sayısıyla para kaybettirir
- ATR Stop: Max DD'yi kontrol altına almak için zorunlu
- Gerçek veri: sentetik testten sonra gerçek BIST davranışını doğrula

In [ ]:
import subprocess, sys
for pkg in ["pandas","numpy","scipy","scikit-learn","xgboost","arch","hmmlearn","matplotlib","yfinance"]:
    r = subprocess.run([sys.executable,"-m","pip","install","-q",pkg], capture_output=True)
    print(f"  {pkg}: {'OK' if r.returncode==0 else 'HATA'}")
print("\nHazır.")

In [ ]:
import warnings, abc, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

# ══════════════════════════════════════════════════════════
# KULLANıCı AYARLARI — Buradan özelleştirin
# ══════════════════════════════════════════════════════════
HISSE_LISTESI = ["THYAO.IS", "GARAN.IS", "ASELS.IS", "BIMAS.IS", "EREGL.IS"]
ENDEKS        = "^XU100"
VERI_PERIYODU = "8y"          # Geçmiş veri uzunluğu
PORTFOY_TL    = 100_000       # Toplam portföy büyüklüğü (TL)
KELLY_FRAKSIYON = 0.25        # Quarter Kelly (güvenli)
MAX_POZISYON_PCT= 0.15        # Maksimum pozisyon: portföyün %15'i
MAX_DD_LIMIT    = 0.25        # Circuit breaker: %25 DD'de durdur

# Walk-Forward parametreleri
WF_TRAIN_MIN = 504   # Minimum eğitim: 2 yıl
WF_TEST_DAYS = 63    # Test foldu: ~3 ay
WF_STEP      = 63

# Teknik gösterge parametreleri
RSI_P = 14; EMA_F,EMA_S,EMA_SIG = 12,26,9; VOL_W = 20
GARCH_LQ,GARCH_HQ = 0.35,0.70; MOM_W = 5
ATR_MULTIPLIER = 2.0   # Stop-loss mesafesi: ATR'nin 2 katı
RR_TARGET      = 2.0   # Hedef Risk:Reward oranı

FEATURES = ["RSI","MACD","MACD_hist","MACD_signal","Vol_ratio","Rel_strength",
            "ret_1d","ret_5d","ret_20d","ATR_pct","BB_zscore","price_pos"]

print("Konfigürasyon yüklendi.")
print(f"  Hisseler  : {HISSE_LISTESI}")
print(f"  Portföy   : {PORTFOY_TL:,} TL")
print(f"  Kelly frak: {KELLY_FRAKSIYON}")
print(f"  Max pos   : %{MAX_POZISYON_PCT*100:.0f}")

## Adım 1 — Veri Yükleme (yfinance + Sentetik Yedek)

In [ ]:
def load_bist_data(ticker: str, endeks: str = ENDEKS, period: str = VERI_PERIYODU) -> pd.DataFrame:
    """
    yfinance ile BIST hissesi ve XU100 endeks verisi çeker.
    Bağlantı sorunu olursa gerçekçi sentetik veri oluşturur.
    """
    import yfinance as yf

    try:
        print(f"  {ticker} yükleniyor...", end=" ")
        h  = yf.download(ticker, period=period, auto_adjust=True, progress=False, timeout=20)
        xe = yf.download(endeks, period=period, auto_adjust=True, progress=False, timeout=20)

        if isinstance(h.columns,  pd.MultiIndex): h.columns  = h.columns.get_level_values(0)
        if isinstance(xe.columns, pd.MultiIndex): xe.columns = xe.columns.get_level_values(0)

        if len(h) < 200:
            raise ValueError(f"Yetersiz veri: {len(h)} satır")

        df = h[["Open","High","Low","Close","Volume"]].copy()
        df["Endeks_Close"] = xe["Close"]
        df.dropna(inplace=True)
        df.index.name = "Date"
        print(f"OK → {len(df)}g ({str(df.index[0])[:10]} → {str(df.index[-1])[:10]})")
        return df, ticker

    except Exception as e:
        print(f"HATA ({e}) → Sentetik veri kullanılıyor")
        return _synthetic_fallback(ticker), ticker + "_SYN"


def _synthetic_fallback(ticker, n=2000, seed=None):
    """yfinance başarısız olursa gerçekçi sentetik veri üretir."""
    seed = seed or abs(hash(ticker)) % 10000
    rng  = np.random.default_rng(seed)
    mu   = 8e-4; s0 = 0.019
    eps  = rng.standard_normal(n)
    vol  = np.zeros(n); vol[0] = s0
    for t in range(1, n):
        vol[t] = np.sqrt(max(1e-6, s0**2*0.04 + 0.09*(vol[t-1]*eps[t-1])**2 + 0.87*vol[t-1]**2))
    regime = np.ones(n); ib = False; rng2 = np.random.default_rng(seed+1)
    for t in range(100, n):
        if not ib and rng2.random() < 0.002:  ib = True
        elif ib and rng2.random() < 0.008:    ib = False
        if ib: regime[t] = -0.4
    lr    = regime * mu + vol * eps
    c     = 50 * np.exp(np.cumsum(lr))
    intra = 0.013
    h_    = c * np.exp( abs(rng.normal(0, intra, n)))
    l_    = c * np.exp(-abs(rng.normal(0, intra, n)))
    o_    = c * np.exp(rng.normal(0, intra*0.4, n))
    vol_  = (700_000*(1+abs(lr)/s0)*rng.lognormal(0,0.4,n)).astype(int)
    idx_lr= 0.65*lr + 0.35*(5e-4+0.013*rng.standard_normal(n))
    endeks= 8500*np.exp(np.cumsum(idx_lr))
    dates = pd.bdate_range("2016-01-04", periods=n, freq="B")
    df = pd.DataFrame({"Open":o_,"High":h_,"Low":l_,"Close":c,"Volume":vol_,"Endeks_Close":endeks},index=dates)
    df.index.name = "Date"
    return df


# Tüm hisseleri yükle
DATA = {}
for ticker in HISSE_LISTESI:
    df_raw, label = load_bist_data(ticker)
    DATA[label] = df_raw

print(f"\nYüklenen hisse sayısı: {len(DATA)}")

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_index(); c = df["Close"]
    df["log_ret"] = np.log(c / c.shift(1))
    df["Label"]   = (df["log_ret"].shift(-1) > 0).astype(float)
    d = c.diff()
    df["RSI"]     = 100-(100/(1+d.where(d>0,0).rolling(RSI_P).mean()/((-d).where(d<0,0).rolling(RSI_P).mean()+1e-9)))
    df["MACD"]    = c.ewm(span=EMA_F,adjust=False).mean()-c.ewm(span=EMA_S,adjust=False).mean()
    df["MACD_signal"] = df["MACD"].ewm(span=EMA_SIG,adjust=False).mean()
    df["MACD_hist"]   = df["MACD"]-df["MACD_signal"]
    df["Vol_ratio"]   = df["Volume"]/(df["Volume"].rolling(VOL_W).mean()+1e-9)
    df["Rel_strength"]= c/(df["Endeks_Close"]+1e-9)
    df["ret_1d"]  = df["log_ret"]
    df["ret_5d"]  = np.log(c/c.shift(5))
    df["ret_20d"] = np.log(c/c.shift(20))
    tr  = pd.concat([df["High"]-df["Low"],(df["High"]-c.shift(1)).abs(),(df["Low"]-c.shift(1)).abs()],axis=1).max(axis=1)
    df["ATR_raw"] = tr.rolling(14).mean()
    df["ATR_pct"] = df["ATR_raw"] / c
    bb_m = c.rolling(20).mean(); bb_s = c.rolling(20).std()+1e-9
    df["BB_zscore"] = (c-bb_m)/bb_s
    h20 = df["High"].rolling(20).max(); l20 = df["Low"].rolling(20).min()
    df["price_pos"] = (c-l20)/(h20-l20+1e-9)
    df.dropna(inplace=True)
    return df

## Adım 2 — Strateji Sınıfları (WFO Doğrulanmış Top 2 + Ensemble)

In [ ]:
# ════════════════════════════════════════════════════
# STRATEJİ 1: HMM (Jim Simons — Gizli Markov Modeli)
# ════════════════════════════════════════════════════
class HMMStrategy:
    name = "HMM"
    def __init__(self, n_states=3):
        self.n = n_states; self.model = None; self.bull = None
        self._mu = None; self._sg = None

    def _feats(self, df):
        r  = df["log_ret"].values
        v5 = pd.Series(r).rolling(5).std().bfill().values
        vr = df["Vol_ratio"].fillna(1).values
        rs = df["Rel_strength"].fillna(df["Rel_strength"].mean()).values
        return np.column_stack([r, v5, vr, rs])

    def fit(self, df):
        from hmmlearn.hmm import GaussianHMM
        X  = self._feats(df)
        self._mu = X.mean(0); self._sg = X.std(0)+1e-9
        Xs = (X-self._mu)/self._sg
        self.model = GaussianHMM(n_components=self.n, covariance_type="full",
                                  n_iter=200, random_state=42)
        self.model.fit(Xs)
        st  = self.model.predict(Xs)
        ret = df["log_ret"].values
        sr  = {s: ret[st==s].mean() if (st==s).sum()>0 else -999 for s in range(self.n)}
        self.bull = max(sr, key=sr.get)

    def predict(self, df) -> pd.Series:
        try:
            X  = self._feats(df)
            Xs = (X-self._mu)/self._sg
            st = self.model.predict(Xs)
            # Olasılık — bull durumuna ait
            pr = self.model.predict_proba(Xs)[:, self.bull]
            pos = (st == self.bull).astype(int)
            return pd.DataFrame({"pos": pos, "prob": pr}, index=df.index)
        except:
            return pd.DataFrame({"pos": np.zeros(len(df),int), "prob": np.full(len(df),0.5)},
                                 index=df.index)

    def current_regime_prob(self, df) -> tuple:
        """Son bar için rejim adını ve bull olasılığını döndürür."""
        from hmmlearn.hmm import GaussianHMM
        res = self.predict(df.iloc[[-1]])
        proba = float(res["prob"].iloc[0])
        label = "Bull" if res["pos"].iloc[0]==1 else ("Bear" if proba < 0.3 else "Sideways")
        return label, proba


# ════════════════════════════════════════════════════
# STRATEJİ 2: AdaptiveMomentum (Medallion ilhamlı)
# ════════════════════════════════════════════════════
class AdaptiveMomStrategy:
    name = "AdaptMom"
    def __init__(self): self._lo = None; self._hi = None

    def fit(self, df):
        rv = df["log_ret"].rolling(20).std()
        self._lo = float(rv.quantile(0.33))
        self._hi = float(rv.quantile(0.67))

    def predict(self, df) -> pd.DataFrame:
        rv  = df["log_ret"].rolling(10).std().values
        r5  = df["ret_5d"].values
        r10 = np.log(df["Close"]/df["Close"].shift(10)).values
        r20 = df["ret_20d"].values
        rsi = df["RSI"].values
        pos = np.zeros(len(df), dtype=int)
        score_arr = np.zeros(len(df))

        for i in range(len(df)):
            v  = rv[i] if not np.isnan(rv[i]) else self._hi
            m5  = r5[i]  if not np.isnan(r5[i])  else 0
            m10 = r10[i] if not np.isnan(r10[i]) else 0
            m20 = r20[i] if not np.isnan(r20[i]) else 0
            if v < self._lo:
                sc = 0.2*m5 + 0.3*m10 + 0.5*m20
            elif v > self._hi:
                sc = 0.6*m5 + 0.3*m10 + 0.1*m20
                if not np.isnan(rsi[i]) and rsi[i] > 70: sc = -abs(sc)
            else:
                sc = 0.4*m5 + 0.4*m10 + 0.2*m20
            score_arr[i] = sc
            pos[i] = 1 if sc > 0 else 0

        # Normalize skoru 0-1 olasılığa çevir (sigmoid benzeri)
        sc_max = np.abs(score_arr).max() + 1e-9
        prob = 0.5 + 0.5 * score_arr / sc_max
        return pd.DataFrame({"pos": pos, "prob": np.clip(prob,0,1)}, index=df.index)


# ════════════════════════════════════════════════════
# STRATEJİ 3: Ensemble (HMM + AdaptMom)
# ════════════════════════════════════════════════════
class EnsembleStrategy:
    name = "Ensemble"
    def __init__(self):
        self.hmm  = HMMStrategy()
        self.adap = AdaptiveMomStrategy()

    def fit(self, df):
        self.hmm.fit(df)
        self.adap.fit(df)

    def predict(self, df) -> pd.DataFrame:
        h = self.hmm.predict(df)
        a = self.adap.predict(df)
        # Ağırlıklı ortalama olasılık: HMM %55, AdaptMom %45
        prob = 0.55 * h["prob"].values + 0.45 * a["prob"].values
        pos  = (prob > 0.5).astype(int)
        return pd.DataFrame({"pos": pos, "prob": prob}, index=df.index)


STRATEGIES = {
    "HMM"      : HMMStrategy(),
    "AdaptMom" : AdaptiveMomStrategy(),
    "Ensemble" : EnsembleStrategy(),
}
print("3 strateji hazır: HMM | AdaptMom | Ensemble")

## Adım 3 — Walk-Forward Motoru + Risk Metrikleri

In [ ]:
def walk_forward(df_feat, strategies, train_min=WF_TRAIN_MIN,
                 test_days=WF_TEST_DAYS, step=WF_STEP) -> dict:
    """Expanding-window WFO. Çıktı: {strateji: {pos, prob} DataFrame}"""
    n   = len(df_feat)
    pos_arrs  = {nm: np.zeros(n, dtype=int) for nm in strategies}
    prob_arrs = {nm: np.full(n, 0.5) for nm in strategies}
    fold = 0; te = train_min; t0 = time.time()
    total = (n - train_min) // step

    while te + test_days <= n:
        t1     = min(te + test_days, n)
        tr_df  = df_feat.iloc[:te]
        te_df  = df_feat.iloc[te:t1]
        for nm, strat in strategies.items():
            try:
                strat.fit(tr_df)
                res = strat.predict(te_df)
                pos_arrs[nm][te:t1]  = res["pos"].values[:t1-te]
                prob_arrs[nm][te:t1] = res["prob"].values[:t1-te]
            except Exception as e:
                pass
        fold += 1; te += step
        if fold % 10 == 0 or fold <= 2:
            print(f"  Fold {fold:3d}/{total} | {time.time()-t0:.0f}s", flush=True)

    print(f"  ✓ {fold} fold | {time.time()-t0:.1f}s")
    return {nm: pd.DataFrame({"pos": pos_arrs[nm], "prob": prob_arrs[nm]},
                              index=df_feat.index)
            for nm in strategies}


def calc_wfo_metrics(pos_series, log_ret, wf_start, n_boot=500):
    """Tam metrik paketi — WFO test döneminde."""
    p = pos_series.iloc[wf_start:].shift(1).fillna(0)
    r = log_ret.iloc[wf_start:]
    s = p * r
    total  = float(np.expm1(s.sum())*100)
    std_   = s.std()
    sharpe = float(s.mean()/std_*np.sqrt(252)) if std_>1e-9 else 0.0
    cum    = np.exp(s.cumsum()); pk=cum.cummax()
    maxdd  = float(((cum-pk)/pk).min()*100)
    calmar = total/abs(maxdd) if abs(maxdd)>1e-3 else 0.0
    act    = s[p>0]; wr = float((act>0).mean()*100) if len(act)>0 else 0.0
    # Bootstrap Sharpe CI
    rng = np.random.default_rng(42); arr = s.values
    bsh = [(lambda x: x.mean()/x.std()*np.sqrt(252) if x.std()>1e-9 else 0)
           (rng.choice(arr,size=len(arr),replace=True)) for _ in range(n_boot)]
    ci_lo, ci_hi = np.percentile(bsh,2.5), np.percentile(bsh,97.5)
    t_, pv = (stats.ttest_1samp(act.values,0) if len(act)>10 else (0,1))
    annual = {yr:round(float(np.expm1(g.sum())*100),2) for yr,g in s.groupby(s.index.year)}
    return {
        "Getiri(%)":round(total,2), "Sharpe":round(sharpe,3),
        "CI_lo":round(ci_lo,3), "CI_hi":round(ci_hi,3),
        "MaxDD(%)":round(maxdd,2), "Calmar":round(float(calmar),3),
        "WinRate(%)":round(wr,2), "p-val":round(float(pv),4),
        "_sr":s, "_annual":annual, "_act":act
    }

print("WFO motoru ve metrik fonksiyonları hazır.")

## Adım 4 — Kelly Kriteri & ATR Stop-Loss

In [ ]:
def calc_kelly(active_returns: pd.Series) -> dict:
    """
    WFO geçmişinden Kelly fraksiyonu hesaplar.

    Formül: f* = (p*b - q) / b   |  b = ort_kazanç / ort_kayıp
    Quarter Kelly (k×0.25) uygulanır: daha güvenli, uzun vadede daha stabil.
    """
    wins   = active_returns[active_returns > 0]
    losses = active_returns[active_returns < 0]
    if len(wins) < 5 or len(losses) < 5:
        return {"kelly_raw": 0.0, "kelly_quarter": 0.0, "win_rate": 0.5,
                "payoff_ratio": 1.0, "edge": 0.0}

    p = len(wins) / (len(wins) + len(losses))
    b = float(wins.mean()) / float(abs(losses.mean()) + 1e-9)
    q = 1 - p

    kelly_raw     = max((p * b - q) / (b + 1e-9), 0.0)
    kelly_quarter = min(kelly_raw * KELLY_FRAKSIYON, MAX_POZISYON_PCT)

    edge = p * b - q   # Beklenen getiri / kayıp oranı

    return {
        "kelly_raw"    : round(kelly_raw,   4),
        "kelly_quarter": round(kelly_quarter,4),
        "win_rate"     : round(p, 4),
        "payoff_ratio" : round(b, 3),
        "edge"         : round(edge, 4),
        "avg_win_pct"  : round(float(wins.mean())*100,  3),
        "avg_loss_pct" : round(float(losses.mean())*100,3),
        "n_trades"     : len(wins) + len(losses),
    }


def calc_atr_stop(df: pd.DataFrame, entry_price: float,
                  multiplier: float = ATR_MULTIPLIER) -> dict:
    """
    Son ATR(14) değerinden dinamik stop-loss hesaplar.

    Stop  = entry - multiplier × ATR
    Hedef = entry + RR_TARGET × (entry - stop)
    """
    atr_val   = float(df["ATR_raw"].iloc[-1])
    stop      = entry_price - multiplier * atr_val
    target    = entry_price + RR_TARGET * multiplier * atr_val
    risk_pct  = (entry_price - stop) / entry_price * 100
    reward_pct= (target - entry_price) / entry_price * 100

    return {
        "entry"     : round(entry_price, 2),
        "stop"      : round(stop, 2),
        "target"    : round(target, 2),
        "atr"       : round(atr_val, 4),
        "risk_pct"  : round(risk_pct, 2),
        "reward_pct": round(reward_pct, 2),
        "rr_ratio"  : round(reward_pct / (risk_pct + 1e-9), 2),
    }


def size_position(kelly_data: dict, atr_data: dict,
                  portfolio_tl: float = PORTFOY_TL) -> dict:
    """
    Kelly fraksiyonu × Portföy = Pozisyon TL
    Lot sayısı = Pozisyon TL / Giriş fiyatı
    Risk TL    = Lot × (Giriş - Stop)
    """
    kelly_pct   = kelly_data["kelly_quarter"]
    entry       = atr_data["entry"]
    stop        = atr_data["stop"]

    pos_tl      = kelly_pct * portfolio_tl
    lots        = pos_tl / (entry + 1e-9)
    risk_tl     = lots * (entry - stop)
    risk_pct_pf = risk_tl / portfolio_tl * 100

    return {
        "pozisyon_tl"   : round(pos_tl, 0),
        "lot_sayisi"    : round(lots, 1),
        "risk_tl"       : round(risk_tl, 0),
        "risk_pct_pf"   : round(risk_pct_pf, 2),
        "kelly_pct"     : round(kelly_pct * 100, 2),
    }


def backtest_with_kelly(pos_series, log_ret, kelly_frac, wf_start) -> dict:
    """
    Kelly fraksiyonuyla ölçeklendirilmiş backtest.
    Sabit yüzde (binary sinyal yerine) bir nakit rezerv ayrılır.
    """
    p  = pos_series.iloc[wf_start:].shift(1).fillna(0)
    r  = log_ret.iloc[wf_start:]
    # Kelly ile ölçekleme: pozisyondayken portfolyonun kelly_frac'i riske atılır
    s  = p * kelly_frac * r
    total  = float(np.expm1(s.sum())*100)
    std_   = s.std()
    sharpe = float(s.mean()/std_*np.sqrt(252)) if std_>1e-9 else 0.0
    cum    = np.exp(s.cumsum()); pk=cum.cummax()
    maxdd  = float(((cum-pk)/pk).min()*100)
    calmar = total/abs(maxdd) if abs(maxdd)>1e-3 else 0.0
    return {"Getiri(%)":round(total,2),"Sharpe":round(sharpe,3),
            "MaxDD(%)":round(maxdd,2),"Calmar":round(float(calmar),3),"_sr":s}


print("Kelly + ATR risk yönetimi fonksiyonları hazır.")

## Adım 5 — Tek Hisse Tam WFO Analizi

In [ ]:
def analyze_single_stock(ticker: str, df_raw: pd.DataFrame) -> dict:
    SEP = "═"*70
    print(f"\n{SEP}")
    print(f"  {ticker} — TAM WFO ANALİZİ")
    print(SEP)

    # 1. Özellik mühendisliği
    df_f = feature_engineering(df_raw)
    n    = len(df_f)
    est  = (n - WF_TRAIN_MIN) // WF_STEP
    wf_s = WF_TRAIN_MIN

    if est < 5:
        print(f"  UYARI: Yeterli veri yok ({n}g, {est} fold). Atlanıyor.")
        return None

    print(f"  Veri: {n}g | WFO start: {str(df_f.index[wf_s])[:10]} | ~{est} fold")

    # 2. Walk-Forward
    print(f"  WFO çalışıyor ({est} fold)...", flush=True)
    strat_objs = {nm: strat.__class__() for nm, strat in STRATEGIES.items()}
    wfo = walk_forward(df_f, strat_objs)

    # 3. Metrikler + Kelly
    results = {}; kelly_data_all = {}
    for nm, res_df in wfo.items():
        m = calc_wfo_metrics(res_df["pos"], df_f["log_ret"], wf_s)
        kelly_d = calc_kelly(m.pop("_act"))
        annual  = m.pop("_annual")
        sr      = m.pop("_sr")
        results[nm]       = {**m, **{"Kelly%": round(kelly_d["kelly_quarter"]*100,2),
                                      "Payoff": kelly_d["payoff_ratio"],
                                      "Edge": kelly_d["edge"],
                                      "_sr": sr, "_annual": annual}}
        kelly_data_all[nm] = kelly_d

    # Buy & Hold
    bh_m = calc_wfo_metrics(pd.Series(1,index=df_f.index), df_f["log_ret"], wf_s)
    bh_sr = bh_m.pop("_sr"); bh_m.pop("_annual"); bh_m.pop("_act")
    results["BuyHold"] = {**bh_m, **{"Kelly%":0,"Payoff":0,"Edge":0,
                                      "_sr":bh_sr,"_annual":{}}}

    # 4. Kelly ile backtest karşılaştırması
    print(f"\n  Kelly backtest karşılaştırması:")
    for nm in [k for k in results if k!="BuyHold"]:
        k_frac = kelly_data_all[nm]["kelly_quarter"]
        kb = backtest_with_kelly(wfo[nm]["pos"], df_f["log_ret"], k_frac, wf_s)
        results[nm]["Kelly_Getiri%"] = kb["Getiri(%)"]
        results[nm]["Kelly_Sharpe"]  = kb["Sharpe"]
        results[nm]["Kelly_MaxDD%"]  = kb["MaxDD(%)"]
        results[nm]["_kelly_sr"]     = kb["_sr"]
        print(f"    {nm:<12} Binary: {results[nm]['Getiri(%)']:+7.2f}%  "
              f"Kelly: {kb['Getiri(%)']:+7.2f}%  "
              f"Sharpe: {kb['Sharpe']:.3f}  DD: {kb['MaxDD(%)']:.1f}%")

    # 5. Tablo
    show_cols = ["Getiri(%)","Sharpe","CI_lo","CI_hi","MaxDD(%)","WinRate(%)","p-val",
                 "Kelly%","Payoff","Kelly_Getiri%","Kelly_Sharpe","Kelly_MaxDD%"]
    df_show  = pd.DataFrame({k:{c:v for c,v in results[k].items() if c in show_cols}
                              for k in results}).T
    srt      = sorted([k for k in results if k!="BuyHold"],
                       key=lambda x: results[x].get("Kelly_Getiri%",results[x]["Getiri(%)"]), reverse=True)
    print(f"\n  PERFORMANS TABLOSU ({n-wf_s}g out-of-sample | ~{est} fold)")
    print(f"  B&H: {results['BuyHold']['Getiri(%)']:+.2f}%  Sharpe:{results['BuyHold']['Sharpe']:.3f}")
    print(df_show.loc[srt+["BuyHold"]][show_cols[:10]].to_string())

    # 6. Şampiyon
    champion = max(srt, key=lambda x: results[x].get("Kelly_Getiri%", results[x]["Getiri(%)"]))
    runner_up= [k for k in srt if k!=champion][0] if len(srt)>1 else champion

    print(f"\n  ★ ŞAMPİYON: {champion}  Kelly getiri: {results[champion]['Kelly_Getiri%']:+.2f}%")

    return {
        "ticker": ticker, "df_feat": df_f, "results": results,
        "wfo": wfo, "kelly_data": kelly_data_all,
        "champion": champion, "runner_up": runner_up,
    }

## Adım 6 — Bugünkü Sinyal Raporu

In [ ]:
def generate_signal_report(analysis: dict, portfolio_tl: float = PORTFOY_TL) -> dict:
    """
    Şampiyon stratejiyi TÜM geçmişe fit eder.
    Son günün sinyalini, pozisyon büyüklüğünü ve stop-loss'u hesaplar.
    """
    SEP = "═"*65
    df_f     = analysis["df_feat"]
    champion = analysis["champion"]
    kelly_d  = analysis["kelly_data"][champion]
    ticker   = analysis["ticker"]

    # Şampiyon stratejiyi tüm veriye fit et
    strat    = STRATEGIES[champion].__class__()
    valid_df = df_f.dropna(subset=["Label"])
    strat.fit(valid_df)

    # Son bar sinyali
    last_bar  = df_f.iloc[[-1]]
    pred      = strat.predict(last_bar)
    signal    = int(pred["pos"].iloc[0])
    prob      = float(pred["prob"].iloc[0])
    last_date = str(df_f.index[-1])[:10]
    entry_px  = float(df_f["Close"].iloc[-1])

    # ATR stop-loss ve pozisyon boyutu
    atr_d  = calc_atr_stop(df_f, entry_px)
    pos_d  = size_position(kelly_d, atr_d, portfolio_tl)

    lt     = df_f.iloc[-1]   # Son bar göstergeleri

    # ── Rapor Çıktısı ────────────────────────────────────────────────────────
    print(f"\n{SEP}")
    print(f"  {ticker} — GÜNLÜK SİNYAL RAPORU  ({last_date})")
    print(SEP)
    print(f"  Strateji     : {champion}")
    print(f"  Son Kapanış  : {entry_px:.2f} TL")
    print()

    # Sinyal
    if signal == 1:
        conf_bar = "█" * int(prob * 20) + "░" * (20 - int(prob * 20))
        print(f"  ╔{'═'*51}╗")
        print(f"  ║  SONRAKI GÜN: AL  ▲  LONG {'':>22}║")
        print(f"  ║  Güven: [{conf_bar}] {prob*100:.1f}% {'':>4}║")
        print(f"  ╚{'═'*51}╝")
    else:
        conf_bar = "█" * int((1-prob)*20) + "░" * (20 - int((1-prob)*20))
        print(f"  ╔{'═'*51}╗")
        print(f"  ║  SONRAKI GÜN: FLAT  ▬  NAKİT {'':>19}║")
        print(f"  ║  Düşüş güveni: [{conf_bar}] {(1-prob)*100:.1f}%  ║")
        print(f"  ╚{'═'*51}╝")

    print()
    if signal == 1:
        print(f"  ─── POZİSYON YÖNETİMİ ──────────────────────────────")
        print(f"  Kelly Fraksiyonu : %{kelly_d['kelly_quarter']*100:.1f}  "
              f"(Ham Kelly: %{kelly_d['kelly_raw']*100:.1f})")
        print(f"  Geçmiş Win Rate  : %{kelly_d['win_rate']*100:.1f}  "
              f"Payoff: {kelly_d['payoff_ratio']:.2f}x  "
              f"Edge: {kelly_d['edge']:+.4f}")
        print()
        print(f"  Portföy          : {portfolio_tl:>10,.0f} TL")
        print(f"  Pozisyon Büyüklüğü: {pos_d['pozisyon_tl']:>9,.0f} TL  (%{pos_d['kelly_pct']:.1f})")
        print(f"  Tahmini Lot      : {pos_d['lot_sayisi']:>9.0f} adet")
        print()
        print(f"  GİRİŞ FİYATI     : {atr_d['entry']:>9.2f} TL")
        print(f"  STOP-LOSS        : {atr_d['stop']:>9.2f} TL  ({-atr_d['risk_pct']:+.1f}%)")
        print(f"  HEDEF            : {atr_d['target']:>9.2f} TL  (+{atr_d['reward_pct']:.1f}%)")
        print(f"  R:R Oranı        : {atr_d['rr_ratio']:.1f}:1")
        print(f"  Riske Edilen TL  : {pos_d['risk_tl']:>9,.0f} TL  (Portföy: %{pos_d['risk_pct_pf']:.2f})")

    print()
    print(f"  ─── TEKNİK GÖSTERGELER ({last_date}) ─────────────────")
    indicators = [
        ("RSI(14)",        lt.get("RSI",   float("nan")),  30,  70, ""),
        ("MACD Hist",      lt.get("MACD_hist",float("nan")),0,   0, ""),
        ("Bollinger Z",    lt.get("BB_zscore",float("nan")),-2, 2, ""),
        ("Hacim Oranı",    lt.get("Vol_ratio",float("nan")), 0.5, 2, "x"),
        ("5G Getiri",      lt.get("ret_5d",  float("nan"))*100, -5, 5, "%"),
        ("20G Getiri",     lt.get("ret_20d", float("nan"))*100,-10,10, "%"),
        ("Fiyat Poz(0-1)", lt.get("price_pos",float("nan")), 0.3, 0.7, ""),
        ("ATR %",          lt.get("ATR_pct", float("nan"))*100, 0,  3, "%"),
    ]
    for lbl, val, lo, hi, unit in indicators:
        if np.isnan(val):
            print(f"    {lbl:<18}: N/A")
            continue
        flag = ("🔴 Aşırı" if val > hi else ("🟢 Aşırı Satım" if val < lo else "  Nötr"))
        print(f"    {lbl:<18}: {val:>+8.3f}{unit}  {flag}")

    print(SEP)

    return {
        "ticker": ticker, "signal": signal, "prob": prob,
        "entry": entry_px, "stop": atr_d["stop"], "target": atr_d["target"],
        "position_tl": pos_d["pozisyon_tl"], "lots": pos_d["lot_sayisi"],
        "kelly_pct": kelly_d["kelly_quarter"], "rr": atr_d["rr_ratio"],
    }

In [ ]:
def plot_analysis(analysis: dict):
    """3 panelli strateji analiz grafiği."""
    df_f     = analysis["df_feat"]
    results  = analysis["results"]
    champion = analysis["champion"]
    ticker   = analysis["ticker"]
    wf_s     = WF_TRAIN_MIN

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    COLS = {"HMM":"#9C27B0","AdaptMom":"#FF5722","Ensemble":"#F44336","BuyHold":"#9E9E9E"}

    # ── Panel 1: Kümülatif Getiri (Binary) ───────────────────────────────────
    ax1 = axes[0]
    for nm in [k for k in results if k in COLS]:
        sr  = results[nm]["_sr"]
        cum = np.exp(sr.cumsum())
        pct = (cum.iloc[-1]-1)*100
        lw  = 2.5 if nm==champion else (1.2 if nm!="BuyHold" else 1.0)
        ls  = "-" if nm==champion else (":" if nm=="BuyHold" else "--")
        ax1.plot(cum.index, cum, label=f"{'★ ' if nm==champion else ''}{nm} ({pct:+.1f}%)",
                 color=COLS.get(nm,"gray"), lw=lw, ls=ls)
    ax1.axhline(1,color="black",lw=0.7,ls=":",alpha=0.4)
    ax1.axvline(df_f.index[wf_s],color="red",lw=1,ls="--",alpha=0.5,label="WFO Start")
    ax1.set_title(f"{ticker} — Kümülatif Getiri (WFO Binary)", fontweight="bold")
    ax1.set_ylabel("Portföy (Başlangıç=1)"); ax1.legend(fontsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.2f}"))

    # ── Panel 2: Kelly ile Kümülatif Getiri ──────────────────────────────────
    ax2 = axes[1]
    for nm in [k for k in results if k!="BuyHold" and k in COLS and "_kelly_sr" in results[k]]:
        sr  = results[nm]["_kelly_sr"]
        cum = np.exp(sr.cumsum())
        pct = (cum.iloc[-1]-1)*100
        lw  = 2.5 if nm==champion else 1.2
        ax2.plot(cum.index, cum, label=f"{'★ ' if nm==champion else ''}{nm} ({pct:+.1f}%)",
                 color=COLS.get(nm,"gray"), lw=lw)
    bh_sr = results["BuyHold"]["_sr"]; bh_cum=np.exp(bh_sr.cumsum())
    ax2.plot(bh_cum.index,bh_cum,color=COLS["BuyHold"],lw=1,ls=":",
             label=f"B&H ({(bh_cum.iloc[-1]-1)*100:+.1f}%)")
    ax2.axhline(1,color="black",lw=0.7,ls=":",alpha=0.4)
    ax2.set_title(f"{ticker} — Kelly Pozisyon ile Getiri", fontweight="bold")
    ax2.set_ylabel("Portföy Değeri"); ax2.legend(fontsize=8)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.2f}"))

    # ── Panel 3: Karşılaştırma çubuğu ────────────────────────────────────────
    ax3 = axes[2]
    nms   = [k for k in results if k!="BuyHold" and k in COLS]
    metr  = ["Getiri(%)","Kelly_Getiri%","Kelly_Sharpe"]
    metr_lbl = ["Getiri% (Binary)","Getiri% (Kelly)","Sharpe (Kelly)"]
    x  = np.arange(len(metr)); bw=0.22
    offs = np.linspace(-(len(nms)-1)*bw/2,(len(nms)-1)*bw/2,len(nms))
    for i,nm in enumerate(nms):
        vals = [results[nm].get(m,0) for m in metr]
        b = ax3.bar(x+offs[i],vals,bw,label=nm,color=COLS.get(nm,"gray"),alpha=0.8,edgecolor="white")
        for bar,v in zip(b,vals):
            ax3.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.3,
                     f"{v:.1f}",ha="center",va="bottom",fontsize=7)
    ax3.axhline(0,color="black",lw=0.8)
    ax3.set_xticks(x); ax3.set_xticklabels(metr_lbl,fontsize=8,rotation=10)
    ax3.set_title("Strateji Karşılaştırması", fontweight="bold"); ax3.legend(fontsize=8)

    plt.suptitle(f"{ticker} — WFO Analizi | Şampiyon: {champion}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    path = f"/tmp/{ticker.replace('.','_')}_analiz.png"
    plt.savefig(path, dpi=130, bbox_inches="tight"); plt.show()
    print(f"Grafik: {path}")

## Adım 7 — Çoklu Hisse Tarama & Karşılaştırma

In [ ]:
def run_multi_stock_scan(data_dict: dict) -> pd.DataFrame:
    """
    Tüm hisseler için WFO + sinyal çalıştırır.
    En iyi AL sinyali olan hisseler öne çıkarılır.
    """
    all_analyses = {}
    all_signals  = []

    for ticker, df_raw in data_dict.items():
        print(f"\n{'─'*60}")
        analysis = analyze_single_stock(ticker, df_raw)
        if analysis is None:
            continue
        all_analyses[ticker] = analysis
        sig = generate_signal_report(analysis, PORTFOY_TL)
        all_signals.append({
            "Hisse"       : ticker,
            "Sinyal"      : "AL 🔑" if sig["signal"]==1 else "FLAT",
            "Güven%"      : round(sig["prob"]*100,1) if sig["signal"]==1 else round((1-sig["prob"])*100,1),
            "Kelly%"      : round(sig["kelly_pct"]*100,2),
            "Pozisyon TL" : round(sig["position_tl"],0),
            "Stop TL"     : sig["stop"],
            "Hedef TL"    : sig["target"],
            "R:R"         : sig["rr"],
            "WFO Şampiyon": analysis["champion"],
            "Kelly Getiri%":analysis["results"][analysis["champion"]].get("Kelly_Getiri%",0),
        })

    # Özet tablo
    df_scan = pd.DataFrame(all_signals).sort_values(
        ["Sinyal","Güven%"], ascending=[True, False]).reset_index(drop=True)

    print("\n" + "═"*70)
    print("  ÇOKLU HİSSE TARAMA ÖZETI")
    print("═"*70)
    print(df_scan.to_string(index=False))
    print("═"*70)

    al_sinyali = df_scan[df_scan["Sinyal"].str.startswith("AL")]
    if len(al_sinyali) > 0:
        print(f"\n  AL sinyali veren hisse sayısı: {len(al_sinyali)}")
        toplam_pos = al_sinyali["Pozisyon TL"].sum()
        print(f"  Toplam pozisyon büyüklüğü   : {toplam_pos:,.0f} TL "
              f"(Portföy: %{toplam_pos/PORTFOY_TL*100:.1f})")
    else:
        print("\n  Hiçbir hissede AL sinyali yok. Piyasa bekleme döneminde olabilir.")

    return df_scan, all_analyses

In [ ]:
def plot_portfolio_signals(df_scan: pd.DataFrame, all_analyses: dict):
    """Portföy sinyal özet grafiği."""
    if not all_analyses:
        print("Analiz verisi yok."); return

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # ── Panel 1: Hisse Kelly Getirileri ─────────────────────────────────────
    ax1 = axes[0]
    tickers = list(all_analyses.keys())
    colors  = ["#4CAF50" if df_scan.loc[df_scan["Hisse"]==t,"Sinyal"].values[0].startswith("AL")
               else "#FF5722" for t in tickers if t in df_scan["Hisse"].values]
    kelly_rets = [all_analyses[t]["results"][all_analyses[t]["champion"]].get("Kelly_Getiri%",0)
                  for t in tickers]
    ax1.barh(tickers, kelly_rets, color=colors[:len(tickers)], edgecolor="white", alpha=0.85)
    ax1.axvline(0, color="black", lw=0.8)
    ax1.set_xlabel("Kelly Pozisyon ile WFO Getiri (%)")
    ax1.set_title("Hisse Başı WFO Getiri (Kelly)", fontweight="bold")
    for i, v in enumerate(kelly_rets):
        ax1.text(v + (1 if v>=0 else -1), i, f"{v:+.1f}%", va="center", fontsize=8)

    # ── Panel 2: Sinyal & Güven ───────────────────────────────────────────────
    ax2 = axes[1]
    al_df   = df_scan[df_scan["Sinyal"].str.startswith("AL")].head(10)
    flat_df = df_scan[df_scan["Sinyal"]=="FLAT"].head(5)
    shown   = pd.concat([al_df, flat_df])
    bar_c   = ["#4CAF50" if s.startswith("AL") else "#607D8B" for s in shown["Sinyal"]]
    ax2.barh(shown["Hisse"], shown["Güven%"], color=bar_c, alpha=0.85, edgecolor="white")
    ax2.axvline(50, color="black", lw=0.8, ls="--", alpha=0.5)
    ax2.set_xlabel("Sinyal Güven Skoru (%)"); ax2.set_xlim(0, 100)
    ax2.set_title("Sinyal Güven Dağılımı", fontweight="bold")

    plt.suptitle("Çoklu Hisse Portföy Sinyal Özeti", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/tmp/portfoy_sinyal_ozeti.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Grafik: /tmp/portfoy_sinyal_ozeti.png")

## Çalıştır — Tam Sistem

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  TAM SİSTEMİ ÇALIŞTIR
#  Beklenen süre: ~5-15 dk (yüklenen hisse sayısına ve veri uzunluğuna göre)
# ════════════════════════════════════════════════════════════════════════

# ADIM 1: Tüm hisseleri tara
df_scan, all_analyses = run_multi_stock_scan(DATA)

# ADIM 2: Her hisse için grafik
for ticker, analysis in all_analyses.items():
    plot_analysis(analysis)

# ADIM 3: Portföy sinyal özet grafiği
plot_portfolio_signals(df_scan, all_analyses)

# ── Özet ────────────────────────────────────────────────────────────────────
print("\n" + "═"*70)
print("  SİSTEM TAMAMLANDI")
print("═"*70)
print(f"  Analiz edilen hisse: {len(all_analyses)}")
al_count = len(df_scan[df_scan['Sinyal'].str.startswith('AL')])
print(f"  AL sinyali veren   : {al_count}")
print(f"  FLAT (bekle)       : {len(df_scan)-al_count}")
print()
print("  Kelly pozisyon boyutlandırma ne sağlıyor?")
print("  → Aynı strateji, sabit lot yerine dinamik lot → MaxDD düşüyor")
print("  → Edge pozitif olduğunda büyük, negatif dönemde küçük pozisyon")
print()
print("  Sonraki adımlar için öneri:")
print("  1. Gerçek işlem komisyonu ekle (Borsa İstanbul ~0.01-0.1%)")
print("  2. Slippage modeli: ATR'nin %10'u kadar kayma varsay")
print("  3. Birden fazla hissede pozisyon varsa korelasyon limiti uygula")
print("="*70)